# Calcite–dolomite 1-D advection benchmark with xGEM2MT

This is the xGEMS example [`demo_reactive_transport.py`](https://github.com/gemshub/xgems/blob/master/demos/demo_reactive_transport.py) rewritten with xGEM2MT. The benchmark comes from Shao et al. (2009), *Appl. Geochem.* 24, 1287–1300.

A 0.5 m column filled with calcite is flushed with a 1 mmol/L MgCl$_2$ solution. Calcite dissolves, dolomite forms for a while behind the calcite front, and later dissolves again.

xGEM2MT runs this in mode **'A'**: 1-D advection–dispersion with finite differences. At each time step the dissolved elements are moved between nodes, and then all nodes are equilibrated with GEMS3K. In the xGEMS demo, the transport solver and this coupling loop are written in Python.

The input files in `CalciteColumn/` are the xGEMS demo's `CalciteBC` (inflowing fluid, initial system 0) and `CalciteIC` (calcite column, initial system 1).

Both files contain 100 mol of an inert solid (element and phase `Inert`, molar volume 0.0213 L/mol; in the xGEMS benchmark files it is 130.8 mol of tin, Sn), which stands for the solid matrix of the rock: it gives each node a volume of 3.13 L with 1 L of water, i.e. porosity 0.32. Mode 'A' does not need it for the transport (it moves only the dissolved amounts, and the porosity is a transport value): without it the results are the same. With it, GEM needs 3 to 5 times fewer iterations (the number of moles counts: the more moles, the fewer iterations), but the trace amounts of a cold start (AIA) can differ by about 0.2 %.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from xgem2mt import MassTransport

## Collect results during the run

The monitoring function is called after each time step. Here we keep the dissolved Ca, Mg and Cl concentrations (mol/kg) and the amounts of calcite and dolomite in all nodes.

In [ ]:
results = {}

def monitor(engine, step, time):
    nodes = range(engine.nNodes())
    aq = [engine.elementAmountsInPhase(n, "aq_gen") / engine.phaseMass(n, "aq_gen") for n in nodes]
    results[time] = {
        "Ca": [c[engine.indexElement("Ca")] for c in aq],
        "Mg": [c[engine.indexElement("Mg")] for c in aq],
        "Cl": [c[engine.indexElement("Cl")] for c in aq],
        "Calcite":  [engine.phaseAmount(n, "Calcite") for n in nodes],
        "Dolomite": [engine.phaseAmount(n, "Dolomite-dis") for n in nodes],
    }

## Set up the task

The same parameters as in the xGEMS demo: 80 nodes over 0.5 m, porosity 0.32, pore velocity 9.375·10$^{-6}$ m/s and dispersivity 0.0067 m.

All functions and their parameters: [MassTransport in Python](https://github.com/gemshub/xgem2mt/blob/master/docs/python-mass-transport.md).

### Names of the transport values

Some names look alike but are different values. This notebook uses only the names in the first column:

| Name | Symbol | Unit | Meaning | All nodes (before `initNodes`) | Some nodes (`TransportProperties`) |
|---|---|---|---|---|---|
| velocity | v | m/s | pore velocity of the water | `setFluidAdvectionVelocity` | `velocity` (changes only the dispersion) |
| porosity | φ | – | pore volume / bulk volume | `setInitialNodeEffectivePorosity` | `porosity` |
| **dispersivity** | α | m | a length of the porous medium: how much the flow spreads the solutes | `setInitialDispersivity` | `dispersivity` |
| **diffusivity** | D<sub>w</sub> | m²/s | diffusion coefficient of the solutes in water | `setInitialDiffusivity` | `diffusivity` |
| tortuosity | τ | – | how much longer the diffusion path is than the column | `setInitialTortuosityFactor` | `tortuosity` |
| **dispersion coefficient** | D | m²/s | the program calculates it: D = α·v + D<sub>w</sub>·φ/τ | – | – |
| node length | Δx | m | column length / number of nodes | `setSpatialDimensions`, `MassTransport(mode, n)` | – |
| time step | Δt | s | the program calculates it (see the comment at `setTau`) | – | – |

Here the diffusivity is 0, so D = α·v = 0.0067 m × 9.375·10⁻⁶ m/s = 6.3·10⁻⁸ m²/s. Porosity, diffusivity and tortuosity change only the diffusion part of D.

Not used in mode 'A', thus not set here: `setInitialTotalNodeVolume`, `setInitialEffectivePermeability`, `setTypeFluxPhase`.

In [ ]:
nnodes = 80

# (1) Create the MassTransport simulator in 'A' mode (1-D advection, finite differences)
mt_task = MassTransport("A", nnodes)
mt_task.setName("Calcite-dolomite 1-D advection benchmark")

# (2) Time: start, end, step, in s. The program calculates the time step itself, as the smaller of
#       0.5 * node length / velocity                       advection limit:  333 s here
#       node length**2 / (4 * dispersion coefficient)      dispersion limit: 155 s here
#     (node length 0.00625 m, dispersion coefficient 6.3e-8 m2/s: see the table of names above).
#     The run stops at the first step at or after 'end' (here 21144 s), or after (end - start)/step + 1
#     steps. Keep 'step' smaller than the time step, or the run stops too early.
mt_task.setTau(0.0, 21000.0, 10.0)
mt_task.setSpatialDimensions(0.5)            # column length, m; node length = column length / nnodes (y, z: not used)
mt_task.setSIA("+")  # GEM start in each node: "+" from the previous step (SIA, ~5x faster), "-" from scratch (AIA)
mt_task.setOutput(False)

# Transport values for all nodes (names: see the table above)
mt_task.setTimeStepReductionFactor(1.0)          # the time step is divided by this factor; 1 = not reduced
mt_task.setFluidAdvectionVelocity(9.375e-6)      # velocity, m/s (pore velocity)
mt_task.setInitialNodeEffectivePorosity(0.32)    # porosity
mt_task.setInitialDispersivity(0.0067)           # dispersivity, m
mt_task.setInitialDiffusivity(0.0)               # diffusivity, m2/s (0: no diffusion)
mt_task.setInitialTortuosityFactor(1.0)          # tortuosity
mt_task.setMonitoring(monitor)

# (3) Read the GEMS3K files: nodes 0 and 1 get the inflowing fluid (system 0), the others the column (system 1)
mt_task.initNodes("CalciteColumn/CalciteColumn-dat.lst")

`print(mt_task)` (or `mt_task.summary()`) shows the setup: the column, the time step that the program uses, which nodes are fluid and column, their properties and phases, and which function changes each value.

In [ ]:
print(mt_task)

## Run the simulation

In [ ]:
import time

start = time.perf_counter()
mt_task.runMassTransport()
elapsed = time.perf_counter() - start

times = sorted(results)
x = (np.arange(nnodes) + 0.5) * 0.5 / nnodes   # node centres, m
end = results[times[-1]]
print(f"Simulated: {len(times)} time steps up to t = {times[-1]:.0f} s of model time")
print(f"Computing time: {elapsed:.1f} s")

## Plot the results

Dissolved Ca, Mg and Cl along the column at the final time:

In [ ]:
for name in ("Ca", "Mg", "Cl"):
    plt.plot(x, end[name], label=name)
plt.xlabel("Distance, m"); plt.ylabel("Concentration, mol/kg"); plt.legend(); plt.show()

Calcite and dolomite at the final time:

In [ ]:
plt.plot(x, end["Calcite"], label="Calcite")
plt.plot(x, end["Dolomite"], label="Dolomite")
plt.xlabel("Distance, m"); plt.ylabel("Amount, mol"); plt.legend(); plt.show()

The dolomite zone moves along the column with time:

In [ ]:
for t in times[::27]:
    plt.plot(x, results[t]["Dolomite"], label=f"{t:.0f} s")
plt.xlabel("Distance, m"); plt.ylabel("Dolomite, mol"); plt.legend(); plt.show()

**Breakthrough curves**: the dissolved amounts against time at fixed points of the column, as in the effluent of a column experiment. The x axis is the number of pore volumes, velocity × time / column length.

This run lasts 0.39 pore volumes, so the reaction front (at about 0.17 m) does not get to the outlet. The plot therefore shows three points inside the column and the outlet. The outlet is node n − 2: the last node keeps its start composition. To compare with the effluent of an experiment, run for at least 1 to 2 pore volumes, e.g. `setTau(0.0, 2 * 0.5 / 9.375e-6, 10.0)` (107 000 s).

In [ ]:
velocity, length = 9.375e-6, 0.5                  # as in the setup above
pore_volumes = np.array(times) * velocity / length
points = {"0.05 m": 8, "0.10 m": 16, "0.15 m": 24, "outlet": nnodes - 2}   # node = position / node length

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, node in points.items():
    axes[0].plot(pore_volumes, [results[t]["Mg"][node] for t in times], label=label)
    axes[1].plot(pore_volumes, [results[t]["Ca"][node] for t in times], label=label)
axes[0].set(xlabel="Pore volumes", ylabel="Mg, mol/kg", title="Mg (from the inflow)")
axes[1].set(xlabel="Pore volumes", ylabel="Ca, mol/kg", title="Ca (from calcite)")
axes[0].legend(); axes[1].legend(); plt.tight_layout(); plt.show()

Animation of the calcite and dolomite fronts over all time steps:

In [ ]:
fig, ax = plt.subplots()
cal, = ax.plot(x, results[times[0]]["Calcite"], label="Calcite")
dol, = ax.plot(x, results[times[0]]["Dolomite"], label="Dolomite")
ax.set(xlabel="Distance, m", ylabel="Amount, mol", ylim=(0, 2.5e-4)); ax.legend()

def update(t):
    cal.set_ydata(results[t]["Calcite"]); dol.set_ydata(results[t]["Dolomite"])
    ax.set_title(f"t = {t:.0f} s")

anim = FuncAnimation(fig, update, frames=times[::2], interval=100)  # every 2nd step
plt.close(fig)
HTML(anim.to_jshtml())